# 감정별 음향 특징 추출 (GPU 배치 버전, Google Colab)

로컬의 `extract_features.py`(MFCC + 피치 + 에너지, CPU 멀티프로세싱)와 같은 특징을 GPU에서
배치로 한 번에 계산해서 더 빠르게 뽑는 노트북입니다. 로컬 작업과 **병행 실행**하는 용도로,
같은 4차년도/5차년도 통합 데이터셋, 7개 감정(`happiness, angry, disgust, fear, neutral, sadness, surprise`)
전부를 대상으로 합니다.

**주의: 여기서 계산하는 피치는 `librosa.pyin`과 정확히 같은 값이 아닙니다.**
- 로컬(`extract_features.py`): `librosa.pyin` — CMNDF + HMM(Viterbi)로 프레임 간 피치를 매끄럽게 추정 (정밀하지만 GPU 배치가 어려움)
- 여기(Colab GPU): **정규화 자기상관(normalized autocorrelation) 피크 기반 피치 추정** — YIN의 차이함수 최소화와 수학적으로 동치인 자기상관 최대화를 FFT로 배치 계산 (HMM 스무딩 없음, 프레임별 독립 추정)
- 순수 자기상관은 아주 작은 lag에서 "신호가 그냥 매끄러워서" 생기는 가짜 고상관(옥타브 오류)에 취약해서,
  ① pre-emphasis 필터로 스펙트럼을 평탄화하고 ② 피치 탐색 상한(FMAX)을 실제 성인 대화 음성 범위(500Hz)로
  제한해서 이 문제를 줄였다. 그래도 로컬 pyin보다 프레임별 잡음(가끔 옥타브 오류)이 더 있을 수 있어서,
  **평균(pitch_mean)보다 중앙값(pitch_median)이 이상치에 덜 흔들려 더 안정적**이다.
- 실제 파일 몇 개로 로컬 pyin 결과와 대조 검증한 결과, 중앙값 기준으로 근접했고(예: 171Hz vs 181.8Hz),
  happiness가 angry/sadness보다 피치가 높게 나오는 등 감정별 상대적 경향도 일치했다. 최종적으로 파일당
  평균/표준편차/중앙값 같은 **요약 통계**만 쓸 거라 이 정도 차이는 대체로 무시할 수 있는 수준이지만,
  정확히 같은 숫자가 필요하면 로컬 결과를 기준으로 삼을 것.

**사용 전 준비**
1. Google Drive에 4차년도/5차년도 오디오 폴더와 메타데이터 csv 업로드 (`DTW_colab.ipynb`와 동일)
2. 아래 `설정` 셀에서 `DRIVE_ROOT` / `DATASETS` 경로를 실제 경로로 수정
3. 런타임 유형을 GPU로 설정 (런타임 > 런타임 유형 변경 > GPU)

**설계 개요**
- MFCC: `torchaudio.transforms.MFCC`로 배치 전체를 한 번에 GPU에서 계산
- 피치: 프레임을 한 번에 펼쳐(unfold) FFT 기반 정규화 자기상관을 배치로 계산, 프레임별 피크 lag를 주파수로 변환
- 에너지: 프레임 RMS를 배치로 계산
- 오디오 길이가 파일마다 달라서 배치 안에서는 최대 길이로 zero-padding하고, 각 파일의 실제 길이에서 나오는 유효 프레임 수만큼만 통계에 반영(패딩 구간은 마스킹으로 제외)
- 세션이 끊겨도 이어갈 수 있도록 감정(situation)별로 처리한 파일 수를 체크포인트로 저장

## 1. Google Drive 마운트 및 GPU 확인

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import torchaudio

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 2. 설정 (실제 경로/파라미터로 수정)

In [ ]:
import os
import time

# TODO: 실제 Drive 경로로 수정
DRIVE_ROOT = '/content/drive/MyDrive/emotion_voice_dataset'

# 4차년도/5차년도 데이터셋을 통합해서 처리한다. DTW_colab.ipynb / 로컬 similarity.py와 동일한 구조.
DATASETS = [
    {
        'audio_dir': os.path.join(DRIVE_ROOT, '4차년도'),
        'sample_csv': os.path.join(DRIVE_ROOT, '4차년도.csv'),
        'encoding': 'cp949',
    },
    {
        'audio_dir': os.path.join(DRIVE_ROOT, '5차년도_2차'),
        'sample_csv': os.path.join(DRIVE_ROOT, '5차년도_2차.csv'),
        'encoding': 'cp949',
    },
]

# 데이터셋마다 상황 라벨 표기가 달라서(예: 4차년도 "anger"/"sad" vs
# 5차년도 "angry"/"sadness") 하나로 맞춰준다.
SITUATION_ALIASES = {
    'anger': 'angry',
    'sad': 'sadness',
}

OUTPUT_DIR = os.path.join(DRIVE_ROOT, 'output', 'features_gpu')
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, 'output', 'features_gpu_checkpoint')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

ORIG_SR = 48000            # 데이터셋 오디오의 표본 주파수 (전 파일 동일. 다르면 배치 로딩 시 개별 리샘플링)
N_MFCC = 13
MFCC_N_FFT = 2048
MFCC_HOP = 512
MFCC_N_MELS = 40

PITCH_SR = 8000             # 피치 추적 전용 다운샘플링 레이트 (사람 목소리 피치 대역을 충분히 커버)
PITCH_FRAME_LENGTH = 1024
PITCH_HOP = 256
FMIN = 65.0                 # ~C2, 성인 남성 저음 하한
FMAX = 500.0                 # 성인 대화 음성의 현실적 상한(너무 높이면 아주 작은 lag에서
                             # '신호가 매끄러워서' 생기는 가짜 고상관에 취약해져 옥타브 오류가 급증한다)
TAU_MIN = int(PITCH_SR / FMAX)
TAU_MAX = int(PITCH_SR / FMIN)
VOICED_THRESHOLD = 0.3        # 정규화 자기상관 피크가 이 값 이상이면 유성음으로 판단

ENERGY_FRAME_LENGTH = 2048
ENERGY_HOP = 512

BATCH_SIZE = 500               # 한 배치에서 동시에 처리할 파일 수 (GPU 메모리에 맞춰 조절)

MFCC_TRANSFORM = torchaudio.transforms.MFCC(
    sample_rate=ORIG_SR,
    n_mfcc=N_MFCC,
    melkwargs={'n_fft': MFCC_N_FFT, 'hop_length': MFCC_HOP, 'n_mels': MFCC_N_MELS},
).to(DEVICE)

## 3. 데이터 로드 (그룹 병합 + wav_id -> 오디오 경로)

In [ ]:
import csv

_WAV_ID_TO_AUDIO_DIR: dict = {}


def load_groups() -> dict:
    groups: dict = {}
    for dataset in DATASETS:
        with open(dataset['sample_csv'], encoding=dataset['encoding'], newline='') as f:
            reader = csv.DictReader(f)
            for row in reader:
                situation = SITUATION_ALIASES.get(row['상황'].strip(), row['상황'].strip())
                wav_id = row['wav_id'].strip()
                groups.setdefault(situation, []).append(wav_id)
                _WAV_ID_TO_AUDIO_DIR[wav_id] = dataset['audio_dir']
    return groups


def get_audio_path(wav_id: str) -> str:
    audio_dir = _WAV_ID_TO_AUDIO_DIR.get(wav_id)
    if audio_dir is None:
        raise KeyError(f"'{wav_id}'의 오디오 폴더를 알 수 없습니다. load_groups()를 먼저 호출했는지 확인하세요.")
    return os.path.join(audio_dir, f'{wav_id}.wav')

## 4. 배치 오디오 로딩 (zero-padding + 길이 마스크)

In [ ]:
import numpy as np
from scipy.io import wavfile


def load_batch_waveforms(wav_ids: list) -> tuple:
    """wav_ids에 대해 파형을 읽어 (B, max_len) 텐서로 패딩.
    sample rate가 ORIG_SR과 다른 파일이 있으면 개별적으로 리샘플링한다.

    반환: waveforms (B, max_len) float32 텐서(DEVICE 위), lengths (원본 표본 수 리스트)
    """
    raw = []
    lengths = []
    for wav_id in wav_ids:
        sr, data = wavfile.read(get_audio_path(wav_id))
        if data.ndim > 1:
            data = data.mean(axis=1)
        data = data.astype(np.float32)
        if data.size and np.abs(data).max() > 1.0:
            data = data / 32768.0  # int16 PCM 정규화
        if sr != ORIG_SR:
            waveform = torch.from_numpy(data).unsqueeze(0)
            waveform = torchaudio.functional.resample(waveform, sr, ORIG_SR).squeeze(0)
            data = waveform.numpy()
        raw.append(data)
        lengths.append(len(data))

    max_len = max(lengths) if lengths else 0
    batch = np.zeros((len(raw), max_len), dtype=np.float32)
    for i, data in enumerate(raw):
        batch[i, : len(data)] = data

    return torch.from_numpy(batch).to(DEVICE), lengths

## 5. 배치 특징 계산 (MFCC / 피치 / 에너지)

In [ ]:
@torch.no_grad()
def batched_mfcc_stats(waveforms: torch.Tensor, lengths: list) -> tuple:
    """waveforms: (B, max_len). 반환: mfcc_mean(B, N_MFCC), mfcc_std(B, N_MFCC) (numpy)."""
    mfcc = MFCC_TRANSFORM(waveforms)  # (B, N_MFCC, frames)
    n_frames = mfcc.shape[-1]

    frame_idx = torch.arange(n_frames, device=DEVICE).unsqueeze(0)  # (1, frames)
    valid_frames = torch.tensor(
        [min(n_frames, 1 + length // MFCC_HOP) for length in lengths], device=DEVICE
    ).unsqueeze(1)  # (B, 1)
    mask = (frame_idx < valid_frames).unsqueeze(1).float()  # (B, 1, frames)

    counts = mask.sum(dim=-1).clamp(min=1)  # (B, 1)
    mean = (mfcc * mask).sum(dim=-1) / counts
    var = ((mfcc - mean.unsqueeze(-1)) ** 2 * mask).sum(dim=-1) / counts
    std = var.clamp(min=0).sqrt()

    return mean.cpu().numpy(), std.cpu().numpy()

In [ ]:
@torch.no_grad()
def batched_pitch_stats(waveforms: torch.Tensor, lengths: list) -> tuple:
    """waveforms: (B, max_len) ORIG_SR 기준.
    반환: pitch_mean, pitch_std, pitch_median, voiced_ratio (모두 (B,) numpy 배열).
    """
    batch_size = waveforms.shape[0]

    resampled = torchaudio.functional.resample(waveforms, ORIG_SR, PITCH_SR)  # (B, max_len_p)
    # pre-emphasis: 저주파 쪽으로 쏠린 스펙트럼을 평탄화해서 아주 작은 lag에서
    # '신호가 그냥 매끄러워서' 생기는 가짜 고상관(옥타브 오류의 주 원인)을 줄인다.
    resampled = torch.cat(
        [resampled[:, :1], resampled[:, 1:] - 0.97 * resampled[:, :-1]], dim=-1
    )
    pitch_lengths = [round(length * PITCH_SR / ORIG_SR) for length in lengths]

    frames = resampled.unfold(-1, PITCH_FRAME_LENGTH, PITCH_HOP)  # (B, frames, PITCH_FRAME_LENGTH)
    n_frames = frames.shape[1]
    if n_frames == 0:
        empty = np.zeros(batch_size, dtype=np.float64)
        return empty, empty, empty, empty

    W = PITCH_FRAME_LENGTH
    nfft = 1
    while nfft < 2 * W:
        nfft *= 2

    # 윈도우 없이(zero-padding만으로) 순수 선형 상호상관을 FFT로 계산한다.
    # acf[..., tau] = sum_{n=0}^{W-1-tau} x[n]*x[n+tau]  (겹치는 구간만의 합, tau=0..W-1)
    spectrum = torch.fft.rfft(frames, n=nfft, dim=-1)
    power = spectrum.real ** 2 + spectrum.imag ** 2
    acf = torch.fft.irfft(power, n=nfft, dim=-1)[..., :W]

    # lag가 커질수록 겹치는 샘플 수가 줄어드는 것을 보정하기 위해, 고정된 acf[0]이 아니라
    # 실제 겹치는 두 구간 각각의 에너지로 정규화한다. (안 그러면 항상 최소 lag가 선택되는
    # 구조적 편향이 생겨 피치가 비정상적으로 높게 나온다.)
    x2 = frames ** 2
    cumsum_x2 = torch.cumsum(x2, dim=-1)
    padded_cumsum = torch.cat(
        [torch.zeros_like(cumsum_x2[..., :1]), cumsum_x2], dim=-1
    )  # (B, frames, W+1), padded_cumsum[..., k] = sum_{n=0}^{k-1} x[n]^2
    total_energy = cumsum_x2[..., -1:]  # (B, frames, 1)

    taus = torch.arange(TAU_MIN, TAU_MAX + 1, device=DEVICE)  # (taus,)
    e1_idx = (W - taus).clamp(min=0, max=W)  # sum_{n=0}^{W-tau-1} x[n]^2 = padded_cumsum[..., W-tau]
    e2_idx = taus.clamp(min=0, max=W)  # sum_{n=tau}^{W-1} x[n]^2 = total - padded_cumsum[..., tau]

    e1 = padded_cumsum[..., e1_idx]  # (B, frames, taus)
    e2 = total_energy - padded_cumsum[..., e2_idx]  # (B, frames, taus)

    candidates = acf[..., TAU_MIN : TAU_MAX + 1] / (e1 * e2).clamp(min=1e-9).sqrt()  # (B, frames, taus)

    peak_val, peak_idx = candidates.max(dim=-1)  # (B, frames)
    tau = peak_idx + TAU_MIN
    f0 = PITCH_SR / tau.float()  # (B, frames)

    frame_idx = torch.arange(n_frames, device=DEVICE).unsqueeze(0)  # (1, frames)
    valid_frames = torch.tensor(
        [min(n_frames, max(0, 1 + (length - PITCH_FRAME_LENGTH) // PITCH_HOP)) for length in pitch_lengths],
        device=DEVICE,
    ).unsqueeze(1)
    valid_mask = frame_idx < valid_frames  # (B, frames)
    voiced_mask = (peak_val >= VOICED_THRESHOLD) & valid_mask

    f0_np = f0.cpu().numpy()
    valid_np = valid_mask.cpu().numpy()
    voiced_np = voiced_mask.cpu().numpy()

    pitch_mean = np.zeros(batch_size, dtype=np.float64)
    pitch_std = np.zeros(batch_size, dtype=np.float64)
    pitch_median = np.zeros(batch_size, dtype=np.float64)
    voiced_ratio = np.zeros(batch_size, dtype=np.float64)

    for i in range(batch_size):
        n_valid = valid_np[i].sum()
        voiced_ratio[i] = voiced_np[i].sum() / n_valid if n_valid > 0 else 0.0
        voiced_f0 = f0_np[i][voiced_np[i]]
        if voiced_f0.size > 0:
            pitch_mean[i] = voiced_f0.mean()
            pitch_std[i] = voiced_f0.std()
            pitch_median[i] = np.median(voiced_f0)

    return pitch_mean, pitch_std, pitch_median, voiced_ratio

In [ ]:
@torch.no_grad()
def batched_energy_stats(waveforms: torch.Tensor, lengths: list) -> tuple:
    """waveforms: (B, max_len). 반환: energy_mean(B,), energy_std(B,) (numpy)."""
    frames = waveforms.unfold(-1, ENERGY_FRAME_LENGTH, ENERGY_HOP)  # (B, frames, ENERGY_FRAME_LENGTH)
    n_frames = frames.shape[1]
    rms = frames.pow(2).mean(dim=-1).clamp(min=0).sqrt()  # (B, frames)

    frame_idx = torch.arange(n_frames, device=DEVICE).unsqueeze(0)
    valid_frames = torch.tensor(
        [min(n_frames, max(0, 1 + (length - ENERGY_FRAME_LENGTH) // ENERGY_HOP)) for length in lengths],
        device=DEVICE,
    ).unsqueeze(1)
    mask = (frame_idx < valid_frames).float()

    counts = mask.sum(dim=-1).clamp(min=1)
    mean = (rms * mask).sum(dim=-1) / counts
    var = ((rms - mean.unsqueeze(-1)) ** 2 * mask).sum(dim=-1) / counts
    std = var.clamp(min=0).sqrt()

    return mean.cpu().numpy(), std.cpu().numpy()

## 6. 체크포인트 유틸

In [ ]:
import json


def checkpoint_path(situation: str) -> str:
    return os.path.join(CHECKPOINT_DIR, f'{situation}.json')


def load_checkpoint(situation: str) -> int:
    path = checkpoint_path(situation)
    if os.path.exists(path):
        with open(path, encoding='utf-8') as f:
            return json.load(f)['files_done']
    return 0


def save_checkpoint(situation: str, files_done: int) -> None:
    with open(checkpoint_path(situation), 'w', encoding='utf-8') as f:
        json.dump({'files_done': files_done}, f)

## 7. 그룹별 처리 메인 루프

In [ ]:
MFCC_MEAN_COLS = [f'mfcc_{i + 1}_mean' for i in range(N_MFCC)]
MFCC_STD_COLS = [f'mfcc_{i + 1}_std' for i in range(N_MFCC)]
FIELDNAMES = (
    ['wav_id', 'situation']
    + MFCC_MEAN_COLS
    + MFCC_STD_COLS
    + ['pitch_mean', 'pitch_std', 'pitch_median', 'voiced_ratio', 'energy_mean', 'energy_std']
)


def process_situation(situation: str, wav_ids: list) -> None:
    n = len(wav_ids)
    files_done = load_checkpoint(situation)
    if files_done >= n:
        print(f"'{situation}' 이미 완료됨 ({n}개). 건너뜀.")
        return

    output_path = os.path.join(OUTPUT_DIR, f'{situation}.csv')
    write_header = not os.path.exists(output_path) or files_done == 0
    mode = 'w' if write_header else 'a'

    print(f"'{situation}': 파일 {n}개, {files_done}개부터 재개")

    with open(output_path, mode, newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        if write_header:
            writer.writerow(FIELDNAMES)

        start = files_done
        t0 = time.perf_counter()
        while start < n:
            end = min(start + BATCH_SIZE, n)
            batch_ids = wav_ids[start:end]

            waveforms, lengths = load_batch_waveforms(batch_ids)
            mfcc_mean, mfcc_std = batched_mfcc_stats(waveforms, lengths)
            pitch_mean, pitch_std, pitch_median, voiced_ratio = batched_pitch_stats(waveforms, lengths)
            energy_mean, energy_std = batched_energy_stats(waveforms, lengths)

            for i, wav_id in enumerate(batch_ids):
                row = (
                    [wav_id, situation]
                    + list(mfcc_mean[i])
                    + list(mfcc_std[i])
                    + [pitch_mean[i], pitch_std[i], pitch_median[i], voiced_ratio[i], energy_mean[i], energy_std[i]]
                )
                writer.writerow(row)

            start = end
            save_checkpoint(situation, start)

            elapsed = time.perf_counter() - t0
            rate = (start - files_done) / elapsed if elapsed > 0 else 0
            remaining = (n - start) / rate if rate > 0 else float('inf')
            print(f"  [{situation}] {start}/{n}개 ({rate:.1f}개/초, 예상 남은 시간 {remaining / 60:.1f}분)")

    print(f"저장 완료: {situation}.csv")


def main() -> None:
    groups = load_groups()
    for situation, wav_ids in groups.items():
        process_situation(situation, wav_ids)


main()

## 참고

- 세션이 끊기면 노트북을 다시 열어 순서대로 셀을 재실행하면 된다. `load_checkpoint`가
  `output/features_gpu_checkpoint/{situation}.json`에 저장된 진행 상황을 읽어 자동으로 이어서 진행한다.
- 로컬 `extract_features.py`(`output/features/per_file_features.csv`)와 컬럼 스키마가 동일해서
  두 결과를 이어붙이거나 감정별 통계를 비교하기 쉽다. 다만 피치 값은 위에서 설명한 대로 계산 방식이
  달라(정규화 자기상관 vs pyin+HMM) 완전히 동일하지는 않다.
- `BATCH_SIZE`를 늘리면 처리율이 올라가지만 GPU 메모리 부족(OOM)이 나면 절반으로 줄여서 재시도한다.
  파일 길이 편차가 크면(예: 11초 vs 2초) zero-padding으로 인한 메모리 낭비가 커지므로, 필요하면
  `load_groups()` 이후 파일 길이순으로 정렬해서 배치를 구성하는 것도 고려할 수 있다.
- 완료 후 `output/features_gpu/*.csv`를 Drive에서 내려받아 로컬 `output/features/per_file_features.csv`와
  병합해서 두 방식의 감정별 통계가 대체로 일치하는지 검증해볼 것을 권장한다.